In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
import default_risk.config as cfg
import logging
import dtale
import dtale.global_state as dtale_global
import gc

dtale_global.cleanup()
gc.collect()

log = logging.getLogger('werkzeug')

column_order_reference="months_balance"

cash_balance_df = pd.read_parquet(cfg.CLEANS_DIR / "POS_CASH_balance_train-cleaned.parquet")

cash_balance_df.sort_values(["id_prev",column_order_reference],inplace=True)

data_frame_size= len(cash_balance_df)

#aux_function

def get_full_sorted_serie_rows(rows : pd.DataFrame) -> pd.DataFrame:
   return recreate_and_sort_series_given_rows(rows,cash_balance_df, "id_prev",column_order_reference)

def get_full_sorted_serie_ids(ids : list) -> pd.DataFrame:
   return recreate_and_sort_the_serie_given_ids(ids,cash_balance_df, "id_prev" ,column_order_reference)

In [6]:
dtale.show(cash_balance_df.head(100))

In [2]:
cash_balance_df["raw_size"]= cash_balance_df.groupby("id_prev").transform("size")

#now, like in all the previous tables, we proceed tu use definition from the EDA.
#For more details consult the markdown in the second celd of eda/eda_cash_balance.ipynb
non_nan_cnt_instalment=  cash_balance_df[cash_balance_df["count_instalment"].notna()]
first_non_nan_cnt_instalment= non_nan_cnt_instalment.groupby("id_prev")["count_instalment"].transform("first")
cash_balance_df["original_expected_duration"] = cash_balance_df["id_prev"].map(first_non_nan_cnt_instalment)

completed_status= cash_balance_df[cash_balance_df["name_contract_status"] == "Completed"]
first_completed_status= completed_status.groupby("id_prev")["count_instalment"].transform("first")
cash_balance_df["factical_duration"] = cash_balance_df["id_prev"].map(first_completed_status)

cash_balance_df["nunique_instalment_future"] = cash_balance_df.where((cash_balance_df["potentaily_on_going"] == 0) & (cash_balance_df["incomplete_sequence"]==0)).groupby("id_prev")["count_instalment_future"].transform("nunique")

cash_balance_df["potentaily_on_going"]= cash_balance_df["potentaily_on_going"].astype(int)

cash_balance_df["diff_expected_real_duration"]= cash_balance_df["original_expected_duration"]  - cash_balance_df["factical_duration"]

cash_balance_df["name_contract_status"]= cash_balance_df["name_contract_status"].str.lower()

cash_balance_df= pd.get_dummies(cash_balance_df,columns=["name_contract_status"])

In [3]:
cash_balance_agg= cash_balance_df.groupby("id_prev").agg({

    "potentaily_on_going" : ["first"],
    "incomplete_sequence" : ["first"],
    "raw_size" : ["first"],
    "original_expected_duration": ["first"],
    "factical_duration": ["first"],
    "diff_expected_real_duration" : ["first"],
    "nunique_instalment_future" : ["first"],

    "amount_advanced_payment": ["max", "sum"],
    "count_instalment" : ["max","min"],
    "count_instalment_future" : ["max","min"],
    "months_balance" : ["max","min"],

    #day_past_due
    "sk_dpd": ["mean","sum","max"],
    "sk_dpd_def": ["mean","sum","max"],

    #categoricals
    "sk_dpd_tecnical": ["mean", "sum"], 
    "sk_dpd_severe": ["mean", "sum"], 
    "dpd_def_tecnical": ["mean", "sum"], 
    "dpd_def_severe": ["mean", "sum"], 
    "name_contract_status_active": ["mean", "sum"],
    "name_contract_status_completed": ["mean", "sum"],
    "flag_is_dead_tail" : ["mean", "sum"],
    "flag_delay_tail" : ["mean", "sum"],
})

cash_balance_agg.columns = [
    f"cash_balance_{col[0]}" if col[1] == "first" else f"cash_balance_{col[0]}_{col[1]}"
    for col in cash_balance_agg.columns
]

def get_time_window(df: pd.DataFrame,time_reference_months) :
    recent_df =df[df["months_balance"] > time_reference_months]
    recent_agg_df=recent_df.groupby("id_curr").agg({
    #"diff_expected_real_duration" : ["mean"],
    "amount_advanced_payment": ["max", "sum"],
    "count_instalment" : ["max","min"],
    "count_instalment_future" : ["max","min"],

    #day_past_due
    "sk_dpd": ["mean","sum","max"],
    "sk_dpd_def": ["mean","sum","max"],

    #categoricals
    "sk_dpd_tecnical": ["mean", "sum"],
    "sk_dpd_severe": ["mean", "sum"],
    "dpd_def_tecnical": ["mean", "sum"],
    "dpd_def_severe": ["mean", "sum"],
    "flag_is_dead_tail" : ["mean", "sum"],
    "flag_delay_tail" : ["mean", "sum"],
})
    time_reference_months= time_reference_months * -1
    recent_agg_df.columns = [
    f"last_{time_reference_months}_cash_balance_{col[0]}_{col[1]}"
    for col in recent_agg_df.columns
    ]

    return recent_agg_df

last_6= get_time_window(cash_balance_df,-6)
last_18= get_time_window(cash_balance_df,-18)

cash_balance_agg.to_parquet(cfg.PROCESSED_DIR / "cash_balance_agg.parquet")
last_6.to_parquet(cfg.PROCESSED_DIR / "cash_balance_time_window.parquet")
last_18.to_parquet(cfg.PROCESSED_DIR / "cash_balance_last_18.parquet")
